# Lecture 2 · Universal Approximation Theorem · a working demo

**Goal** · build a 1-hidden-layer neural network and watch it approximate an arbitrary continuous function.

This is the concrete realization of UAT · no matter how wiggly the target, a wide-enough single-hidden-layer MLP can learn it. We'll target:
1. a sine wave,
2. a step function (harder · discontinuous-ish),
3. a random continuous curve.

Along the way we'll see *why* depth is usually preferred to width in practice.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

## 1 · A shallow MLP · one hidden layer

By UAT · this can in principle approximate any continuous function on `[a, b]`.

In [ ]:
class ShallowMLP(nn.Module):
    def __init__(self, hidden=64, activation=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            activation(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x)

## 2 · Train on a sine wave

Target · `f(x) = sin(2x) + 0.3 sin(5x)`

In [ ]:
def target_fn(x):
    return torch.sin(2 * x) + 0.3 * torch.sin(5 * x)

x_train = torch.linspace(-3, 3, 200).unsqueeze(-1)
y_train = target_fn(x_train)

def train(model, steps=2000, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        y_hat = model(x_train)
        loss = ((y_hat - y_train)**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 400 == 0:
            print(f'step {step:4d}  loss {loss.item():.4f}')
    return model

model = train(ShallowMLP(hidden=64))

with torch.no_grad():
    y_pred = model(x_train)
plt.figure(figsize=(8, 4))
plt.plot(x_train, y_train, label='target', linewidth=2)
plt.plot(x_train, y_pred, label='1 hidden layer · 64 units', linewidth=2, linestyle='--')
plt.legend(); plt.title('UAT with 64 units'); plt.show()

## 3 · How wide do we need?

Sweep hidden width · see the approximation improve.

In [ ]:
widths = [4, 16, 64, 256]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)

for ax, h in zip(axes, widths):
    model = train(ShallowMLP(hidden=h), steps=1500)
    with torch.no_grad():
        y_pred = model(x_train)
    ax.plot(x_train, y_train, linewidth=2, label='target')
    ax.plot(x_train, y_pred, linewidth=2, linestyle='--', label='pred')
    ax.set_title(f'hidden = {h}')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Observation** · with 4 units we get a crude staircase. By 64 units, the approximation is near-perfect. That's UAT in action.

## 4 · Depth vs width · same parameter budget

Is it better to be wide or deep?

Compare three nets with ~matched parameter count:
- wide: 1 hidden × 256 units  ≈ 513 params
- medium: 2 hidden × 32 units  ≈ 1217 params
- deep: 4 hidden × 16 units  ≈ 897 params

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, depth, width):
        super().__init__()
        layers = [nn.Linear(1, width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.ReLU()]
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def n_params(m):
    return sum(p.numel() for p in m.parameters())

wide = DeepMLP(depth=1, width=256)
med  = DeepMLP(depth=2, width=32)
deep = DeepMLP(depth=4, width=16)

print(f'wide  {n_params(wide):5d} params')
print(f'med   {n_params(med):5d} params')
print(f'deep  {n_params(deep):5d} params')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, m) in zip(axes, [('wide', wide), ('med', med), ('deep', deep)]):
    m = train(m, steps=1500)
    with torch.no_grad():
        y_pred = m(x_train)
    ax.plot(x_train, y_train, linewidth=2, label='target')
    ax.plot(x_train, y_pred, linewidth=2, linestyle='--', label='pred')
    err = ((y_pred - y_train)**2).mean().item()
    ax.set_title(f'{name} · MSE {err:.4f}')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Observation** · the deeper net usually reaches lower MSE with similar params. This is why we go deep in practice · the UAT is sufficient but not efficient.

## 5 · What UAT does NOT guarantee

Three important caveats:
1. **Finding** the weights · UAT says such weights exist; SGD may not converge to them.
2. **Generalization** · UAT is on training points; test-set performance is a separate story.
3. **Efficiency** · width needed may be exponential. Depth gives polynomial compression.

Try this: train on 10 points instead of 200 and see how the sweep of widths changes.